<VSCode.Cell language="python">import pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# Shared settings\nsns.set_theme(style='whitegrid')\nplt.rcParams['figure.figsize'] = (10, 6)\n\n# Load processed dataset\ndf_path = 'data/processed/CGD_Transformation_Final.csv'\ndf = pd.read_csv(df_path, parse_dates=['TS'])\n\n# validate data loaded\nprint('Data rows', len(df), 'columns', len(df.columns))\nprint(df[['TS', 'Zone', 'Flow_std', 'Pressure', 'Leak_Alarm']].head())

<VSCode.Cell language="python">hourly = (df.assign(Hour=df['TS'].dt.hour)
            .groupby('Hour')['Flow_std']
            .mean()
            .reset_index())\n\nplt.figure(figsize=(10, 5))\nsns.lineplot(data=hourly, x='Hour', y='Flow_std', marker='o')\nplt.title('Average Hourly Demand Trend (Flow_std)')\nplt.xlabel('Hour of Day')\nplt.ylabel('Average Flow_std (Sm3)')\nplt.xticks(range(0, 24))\nplt.tight_layout()\nplt.show()

<VSCode.Cell language="python">zone_consumption = df.groupby('Zone')['Flow_std'].sum().reset_index().sort_values('Flow_std', ascending=False)\n\nplt.figure(figsize=(12, 6))\nsns.barplot(data=zone_consumption, x='Zone', y='Flow_std', palette='viridis')\nplt.title('Total Flow_std by Zone')\nplt.xlabel('Zone')\nplt.ylabel('Total Flow_std (Sm3)')\nplt.xticks(rotation=45)\nplt.tight_layout()\nplt.show()

<VSCode.Cell language="python">import os\nufg_path = 'data/processed/ufg_dataset.csv'\nif os.path.exists(ufg_path):\n    ufg_df = pd.read_csv(ufg_path)\nelse:\n    # compute fallback\    gas_input = df.groupby('DMA_Zone')['Flow_std'].sum().reset_index(name='Gas_Input_Sm3')\n    gas_billed = df[df['Is_Billable'] == True].groupby('DMA_Zone')['Flow_std'].sum().reset_index(name='Gas_Billed_Sm3')\n    ufg_df = gas_input.merge(gas_billed, on='DMA_Zone', how='left').fillna(0)\n    ufg_df['UFG_Sm3'] = ufg_df['Gas_Input_Sm3'] - ufg_df['Gas_Billed_Sm3']\n    ufg_df['UFG_Percent'] = (ufg_df['UFG_Sm3'] / ufg_df['Gas_Input_Sm3'].replace(0, pd.NA) * 100).round(2)\n\nufg_df_sorted = ufg_df.sort_values('UFG_Percent', ascending=False)\nplt.figure(figsize=(12, 6))\nsns.barplot(data=ufg_df_sorted, x='DMA_Zone', y='UFG_Percent', palette='rocket')\nplt.title('UFG Percent by DMA Zone')\nplt.xlabel('DMA Zone')\nplt.ylabel('UFG Percent (%)')\nplt.xticks(rotation=45)\nplt.tight_layout()\nplt.show()

<VSCode.Cell language="python">pressure_counts = df['Pressure_Violation'].value_counts(dropna=False).reset_index()\npressure_counts.columns = ['Pressure_Violation', 'Count']\n\nplt.figure(figsize=(8, 5))\nsns.barplot(data=pressure_counts, x='Pressure_Violation', y='Count', palette='magma')\nplt.title('Pressure Violation Counts')\nplt.xlabel('Pressure Violation')\nplt.ylabel('Count')\nplt.tight_layout()\nplt.show()\n\n# Also violation count by zone\npressure_zone = df[df['Pressure_Violation'] == True].groupby('Zone').size().reset_index(name='Violation_Count').sort_values('Violation_Count', ascending=False)\nplt.figure(figsize=(12, 6))\nsns.barplot(data=pressure_zone, x='Zone', y='Violation_Count', palette='flare')\nplt.title('Pressure Violation Count by Zone')\nplt.xticks(rotation=45)\nplt.tight_layout()\nplt.show()

In [ ]:
<VSCode.Cell language="markdown">## 6. Demand Forecast\nPlot the 30-day demand forecast produced by the ML pipeline (`data/analytics_outputs/demand_forecast_30d.csv`).